# Hangman — GPU training runs

Training only. Labels are pre-computed and attached as a dataset, so the GPU session
spends its time on the model rather than on CPU-bound oracle work.

**Attach two datasets:** `hangman-src` (the package) and `hangman-data`
(`train.txt` + `labels.tsv`). **Settings → Accelerator → GPU T4 ×2.** Internet off.

`data/test.txt` is deliberately absent from the bundle. The leak guards in
`build_labels.py` and `train.py` abort training if an evaluation word ever reaches
the training set, and they report when they had nothing to check against.

Edit `RUNS` below, then Run All. Each run trains, gates on the golden-word check,
evaluates on three held-out seeds, and writes its checkpoint to `/kaggle/working`.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import glob, os, subprocess, sys, time

_hits = sorted(glob.glob("/kaggle/input/**/hangman/tokenizer.py", recursive=True))
assert _hits, "Attach the 'hangman-src' dataset (Add Input)."
SRC = os.path.dirname(os.path.dirname(_hits[0]))
PKG = os.path.dirname(SRC)
sys.path.insert(0, SRC)

def _find(name):
    hits = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True)
            if not p.startswith(PKG)]
    return sorted(hits)[0] if hits else None

TRAIN  = _find("train.txt")
LABELS = _find("labels.tsv")
if not (TRAIN and LABELS):
    # Say what is actually mounted. The usual cause is a re-imported notebook:
    # re-importing creates a NEW notebook with no inputs attached, so it is easy
    # to add the updated hangman-src and forget hangman-data.
    print("Attached under /kaggle/input:")
    for _top in sorted(glob.glob("/kaggle/input/*")):
        print(f"  {_top}")
        for _sub in sorted(glob.glob(f"{_top}/*"))[:8]:
            print(f"    {_sub}")
    _missing = [n for n, v in [("train.txt", TRAIN), ("labels.tsv", LABELS)] if not v]
    raise AssertionError(
        f"missing {_missing} -- Add Input -> 'hangman-data'. "
        f"BOTH hangman-src and hangman-data must be attached.")

# Guard rail: this notebook must never see evaluation data.
assert _find("test.txt") is None, (
    "test.txt is attached to a TRAINING session. Detach it — training must "
    "never have access to the evaluation set.")

WORK = "/kaggle/working"
os.makedirs(f"{WORK}/experiments", exist_ok=True)
os.chdir(WORK)

import torch
gpu = torch.cuda.is_available()
print(f"package : {PKG}")
print(f"train   : {TRAIN}")
print(f"labels  : {LABELS}")
print(f"device  : {'cuda — ' + torch.cuda.get_device_name(0) if gpu else 'CPU (no GPU!)'}")
if not gpu:
    print("  !! Settings -> Accelerator -> GPU T4 x2, or this will take hours.")

def run(script, *args):
    cmd = [sys.executable, f"{PKG}/scripts/{script}", *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    t = time.time()
    r = subprocess.run(cmd, text=True)
    print(f"[{script} finished in {time.time()-t:.0f}s]", flush=True)
    if r.returncode:
        raise RuntimeError(f"{script} failed with code {r.returncode}")

In [ ]:
# ── What to run ────────────────────────────────────────────────────────────────
# Current recipe: per-position supervision with policy/presence losses off.
# Earlier comparative scores used inconsistent splits and are excluded from
# published evidence. See README.md and reports/kaggle_run.md for settings.
# Additional states read targets directly from words and need no oracle.

STATES_PER_WORD = 25    # 220,300 words -> ~5.5M new states (~190 MB, ~50 s),
                        # plus labels.tsv carried through = ~7.0M rows.
                        # Loading that costs ~5.7 GB; Kaggle gives ~13 GB.
                        # 50/word would be ~9 GB and is not worth the risk.

RUNS = [
    # name                 extra training arguments
    ("wide-512-states",    ["--steps", 40000, "--warmup", 1000,
                            "--dim", 512, "--ff", 2048]),
]

BEST_RECIPE = ["--w-perpos", 1.0, "--w-policy", 0, "--w-presence", 0]
BATCH = 256             # recorded training batch size


In [ ]:
# ── Stage 1b: bulk teacher-free states, then train ────────────────────────────
# --include carries rows belonging to the current training split through.
# Generated states are deduplicated per word but may overlap carried rows.
#
# Written to scratch, not /kaggle/working: a 250 MB intermediate does not belong
# in the notebook's saved output.
TEMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
STATES = f"{TEMP}/states.tsv"

run("build_states.py", "--train", TRAIN, "--out", STATES,
    "--per-word", STATES_PER_WORD, "--include", LABELS)
print(f"states file: {os.path.getsize(STATES)/1e9:.2f} GB", flush=True)

results = {}
for name, extra in RUNS:
    ckpt = f"{WORK}/model_{name}.pt"
    print(f"\n{'='*64}\n{name}\n{'='*64}", flush=True)
    try:
        run("train.py", "--labels", STATES, "--out", ckpt,
            "--batch-size", BATCH, "--seed", 1, *BEST_RECIPE, *extra)
        run("evaluate.py", "--model", ckpt, "--train", TRAIN,
            "--n", 2000, "--seeds", 1, 2, 3,
            "--ledger", "experiments/results.jsonl", "--tag", name)
        results[name] = "ok"
    except Exception as exc:                      # one bad config must not kill the queue
        print(f"!! {name} failed: {exc}", flush=True)
        results[name] = f"failed: {exc}"

print("\n" + "="*64)
for name, status in results.items():
    print(f"  {name:20s} {status}")


In [ ]:
# ── Leaderboard across everything in this session's ledger ────────────────────
import json, collections, math

rows = [json.loads(l) for l in open("experiments/results.jsonl")]
by = collections.defaultdict(list)
for r in rows:
    if r.get("tag"):
        by[r["tag"]].append(r)

print(f"{'config':18s} {'mean':>8s} {'spread':>8s} {'wrong':>8s}   seeds")
for tag, rs in sorted(by.items(), key=lambda kv: -sum(r['win_rate'] for r in kv[1])/len(kv[1])):
    w = [r["win_rate"] for r in rs]
    wrong = sum(r["total_wrong"] for r in rs) // len(rs)
    print(f"{tag:18s} {sum(w)/len(w):7.2%} {max(w)-min(w):8.2%} {wrong:8d}   "
          f"{[f'{x:.2%}' for x in w]}")

print("\nCheckpoints written to /kaggle/working — download the winner from the")
print("Output panel, or Save Version to keep them attached to this notebook.")